# Data Generation Pipeline

Stages 1–5 of the benchmarking pipeline.  Run this notebook **once** (in the
`prexsyn_env` kernel) to produce all variant libraries and scoring CSVs that
`pipeline_analysis.ipynb` consumes.

| Stage | Input | Output |
|-------|-------|--------|
| 1 | `chembl_features.npz` | MolSpec objects + feature arrays |
| 2 | Feature arrays | `seeds_for_methods.json` (PrexSyn top-1 per spec) |
| 3 | Seeds | `baseline/baseline_scores.csv` + `synth_checkpoint.json` |
| 4 | Seeds | `crem/crem_scores.csv` + `synth_checkpoint.json` |
| 5 | Seeds | `libinvent/libinvent_scores.csv` + `synth_checkpoint.json` |

All outputs land in `data/generation/`.  Stages 3–5 each end with a
**Stage 4b** cell that applies the property gate first, then runs
AiZynthFinder only on the passing candidates (cheaper than scoring all).

> **Kernel:** `prexsyn_env` (Python 3.11).
> LibINVENT runs as a `conda run -n libinvent_env` subprocess automatically.

In [ ]:
import os, sys
from pathlib import Path

# Repo root — works whether CWD is project root or notebooks/
ROOT = Path().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src" / "modifications" / "ml_based" / "pipeline"))

# JT-VAE: point to jtvae_env conda interpreter (Windows path) and bundled model
_jtvae_python = Path(r"C:\Users\Acmaro\anaconda3\envs\jtvae_env\python.exe")
_jtvae_model  = ROOT / "src/modifications/ml_based/jt_vae/vendor/mol_opt/main/jt_vae/fast_molvae/vae_model/model.iter-25000"
_jtvae_home   = ROOT / "src/modifications/ml_based/jt_vae/vendor/mol_opt/main/jt_vae"
_jtvae_vocab  = _jtvae_home / "data/zinc/vocab.txt"
os.environ.setdefault("JT_VAE_PYTHON",     str(_jtvae_python))
os.environ.setdefault("JT_VAE_MODEL_PATH", str(_jtvae_model))
os.environ.setdefault("JT_VAE_HOME",       str(_jtvae_home))
os.environ.setdefault("JT_VAE_VOCAB_PATH", str(_jtvae_vocab))
os.environ["JT_VAE_DEVICE"] = "cpu"   # CPU + multithreading faster than GPU for small batches

# ── Data directories ──────────────────────────────────────────────────────────
# Toggle to switch between original and stratified sample.
USE_STRATIFIED_SAMPLE = True

if USE_STRATIFIED_SAMPLE:
    GEN_DIR    = ROOT / "data" / "generation_stratified"
    CHEMBL_NPZ = GEN_DIR / "chembl_features_stratified.npz"
else:
    GEN_DIR    = ROOT / "data" / "generation"
    CHEMBL_NPZ = ROOT / "data" / "chembl_features.npz"

BASELINE_DIR  = GEN_DIR / "baseline"
CREM_DIR      = GEN_DIR / "crem"
LI_DIR        = GEN_DIR / "libinvent"
MMPDB_DIR     = GEN_DIR / "mmpdb"
JTVAE_DIR     = GEN_DIR / "jtvae"
for d in [GEN_DIR, BASELINE_DIR, CREM_DIR, LI_DIR, MMPDB_DIR, JTVAE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Output paths (read by pipeline_analysis.ipynb)
SEEDS_JSON      = GEN_DIR / ("seeds_for_methods_stratified.json" if USE_STRATIFIED_SAMPLE else "seeds_for_methods.json")
BASELINE_SCORES = BASELINE_DIR / "baseline_scores.csv"
BASELINE_CKPT   = BASELINE_DIR / "synth_checkpoint.json"
CREM_SCORES     = CREM_DIR / "crem_scores.csv"
CREM_CKPT       = CREM_DIR / "synth_checkpoint.json"
LI_SCORES       = LI_DIR  / "libinvent_scores.csv"
LI_CKPT         = LI_DIR  / "synth_checkpoint.json"
MMPDB_SCORES    = MMPDB_DIR / "mmpdb_scores.csv"
MMPDB_CKPT      = MMPDB_DIR / "synth_checkpoint.json"
JTVAE_SCORES    = JTVAE_DIR / "jtvae_scores.csv"
JTVAE_CKPT      = JTVAE_DIR / "synth_checkpoint.json"

# Intermediate / scratch paths
LI_SCAFFOLDS_SMI = LI_DIR  / "scaffolds.smi"
LI_DECORATED_CSV = LI_DIR  / "decorated.csv"
LI_CONFIG_JSON   = LI_DIR  / "decorate_config.json"
LI_LOG_DIR       = LI_DIR  / "logs" / "decorate"
LI_LOG_DIR.mkdir(parents=True, exist_ok=True)

# External tools
PREXSYN_URL          = "http://localhost:8011/sample"
AIZYNTHFINDER_CONFIG = ROOT / "data" / "aizynthfinder" / "config.yml"
CREM_DB = Path(os.environ.get(
    "CREM_DB_PATH",
    str(ROOT / "data" / "crem_db" / "chembl33_sa2_f5.db")))

LIB_INVENT_ROOT  = ROOT / "src" / "modifications" / "ml_based" / "Lib-INVENT"
LIB_INVENT_MODEL = LIB_INVENT_ROOT / "trained_models" / "reaction_based.model"
LIB_INVENT_PY    = LIB_INVENT_ROOT / "input.py"

MMPDB_DB = Path(os.environ.get(
    "MMPDB_DB_PATH",
    str(ROOT / "data" / "mmpdb_db" / "chembl_50k.mmpdb")))

# Pipeline parameters
N_SPECS            = 100
N_PREXSYN_SAMPLES  = 256
N_BASELINE_RESAMP  = 128
N_CREM_VARIANTS    = 128
N_LI_DECORATIONS   = 128
N_MMPDB_VARIANTS   = 128
N_JTVAE_VARIANTS   = 128
N_WORKERS          = 4
SYNTH_TIMEOUT      = 180
SKIP_SYNTH_CHECK   = True

# Caching control
USE_CACHED_SEEDS    = True
USE_CACHED_BASELINE = True
USE_CACHED_CREM     = True
USE_CACHED_LI       = True
USE_CACHED_MMPDB    = True
USE_CACHED_JTVAE    = False

# Scoring thresholds
TAU_T_LIST   = [0.6, 0.7, 0.85]
TAU_D        = 0.5
TAU_T_SCREEN = min(TAU_T_LIST)

# Sanity checks
print(f"ROOT       : {ROOT}")
print(f"GEN_DIR    : {GEN_DIR}  ({'stratified' if USE_STRATIFIED_SAMPLE else 'original'})")
print(f"ChEMBL NPZ : {'OK' if CHEMBL_NPZ.exists() else 'MISSING'} ({CHEMBL_NPZ.name})")
print(f"Seeds JSON : {'OK' if SEEDS_JSON.exists() else 'MISSING'} ({SEEDS_JSON.name})")
print(f"AiZynth    : {'OK' if AIZYNTHFINDER_CONFIG.exists() else 'MISSING'}")
print(f"CReM DB    : {'OK' if CREM_DB.exists() else 'MISSING'} ({CREM_DB.name})")
print(f"LI model   : {'OK' if LIB_INVENT_MODEL.exists() else 'MISSING'}")
print(f"mmpdb DB   : {'OK' if MMPDB_DB.exists() else 'MISSING'} ({MMPDB_DB.name})")
print(f"JT-VAE py  : {'OK' if _jtvae_python.exists() else 'MISSING'} ({_jtvae_python})")
print(f"JT-VAE mdl : {'OK' if _jtvae_model.exists() else 'MISSING'} ({_jtvae_model.name})")
print()
print("Method cache status:")
for name, p in [("baseline", BASELINE_SCORES), ("crem", CREM_SCORES),
                ("libinvent", LI_SCORES), ("mmpdb", MMPDB_SCORES), ("jtvae", JTVAE_SCORES)]:
    print(f"  {name:<10}: {'[cached]' if p.exists() else '[missing]'}")


In [2]:
import json, re, subprocess
from concurrent.futures import (
    ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed)
from concurrent.futures.process import BrokenProcessPool

import numpy as np
import pandas as pd
import requests
from rdkit import Chem
from tqdm.notebook import tqdm

from src.evaluation.scoring_v2 import (
    make_spec, score_batch, classify_hits, tanimoto_to_spec)
from src.evaluation.synth_parallel import worker_init, score_one


# ── Stage 4b helper: property gate first, AiZynthFinder second ────────────────
def synth_gate_4b(df_scored: pd.DataFrame, ckpt_path,
                  smiles_col: str = "variant_smiles") -> tuple[dict, int]:
    # Property gate first, AiZynthFinder on passing candidates only.
    # Filters to tanimoto >= TAU_T_SCREEN and desirability >= TAU_D,
    # runs retrosynthesis on that subset, writes ckpt_path.
    # Returns ({smiles: is_solved}, num_evaluated).
    
    # Skip synthesis check if requested
    if SKIP_SYNTH_CHECK:
        mask = (df_scored["tanimoto"] >= TAU_T_SCREEN) & (df_scored["desirability"] >= TAU_D)
        candidates = df_scored.loc[mask, smiles_col].dropna().unique().tolist()
        print(f"[4b] SKIP_SYNTH_CHECK=True: marking {len(candidates):,} passing property gate as synthesizable")
        synth = {s: True for s in candidates}
        with open(ckpt_path, "w") as f:
            json.dump(synth, f)
        return synth, len(candidates)
    
    mask = (df_scored["tanimoto"] >= TAU_T_SCREEN) & (df_scored["desirability"] >= TAU_D)
    candidates = df_scored.loc[mask, smiles_col].dropna().unique().tolist()
    n_all = df_scored[smiles_col].dropna().nunique()
    print(f"[4b] Property gate ({TAU_T_SCREEN}/{TAU_D}): "
          f"{len(candidates):,}/{n_all:,} -> AiZynthFinder")

    if not AIZYNTHFINDER_CONFIG.exists():
        print("     [WARN] Config missing -- marking all synthesizable")
        return {s: True for s in candidates}, len(candidates)

    synth: dict = {}
    _pending = list(candidates)
    _to = SYNTH_TIMEOUT or None

    for _round in range(1, 99):
        if _round > 1:
            print(f"     [restart] round {_round}, {len(_pending)} remaining")
        with ProcessPoolExecutor(
            max_workers=N_WORKERS,
            initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),),
            max_tasks_per_child=50,
        ) as pool:
            futures = {pool.submit(score_one, s): s for s in _pending}
            broken  = False
            with tqdm(total=len(_pending),
                      desc=f"AiZynthFinder (round {_round})", unit="mol") as pbar:
                for fut in as_completed(futures):
                    smi = futures[fut]
                    try:
                        _, solved = fut.result(timeout=_to)
                        pbar.update(1)
                    except FutureTimeoutError:
                        solved = False
                    except BrokenProcessPool:
                        broken = True; break
                    except Exception:
                        solved = False
                    synth[smi] = solved
            if broken:
                pass   # outer loop restarts with remaining

        _pending = [s for s in candidates if s not in synth]
        if not _pending:
            break

    with open(ckpt_path, "w") as f:
        json.dump(synth, f)
    n_solved = sum(synth.values())
    print(f"     Solved: {n_solved}/{len(synth)} "
          f"({100 * n_solved / max(len(synth), 1):.1f}%)")
    return synth, len(candidates)



# ── Stage 4a helper: AiZynthFinder first, property gate second ───────────────
def synth_gate_4a(df_scored: pd.DataFrame, ckpt_path,
                  smiles_col: str = "variant_smiles") -> tuple[dict, int]:
    # Standard ordering: retrosynthesis on ALL candidates, property gate later.
    # Use when you want is_solved recorded for every variant (e.g. for analysis).
    # Returns ({smiles: is_solved}, num_evaluated).
    
    # Skip synthesis check if requested
    if SKIP_SYNTH_CHECK:
        candidates = df_scored[smiles_col].dropna().unique().tolist()
        print(f"[4a] SKIP_SYNTH_CHECK=True: marking all {len(candidates):,} as synthesizable")
        synth = {s: True for s in candidates}
        with open(ckpt_path, "w") as f:
            json.dump(synth, f)
        return synth, len(candidates)
    
    candidates = df_scored[smiles_col].dropna().unique().tolist()
    n_all = len(candidates)
    print(f"[4a] AiZynthFinder on all {n_all:,} variants")
    if not AIZYNTHFINDER_CONFIG.exists():
        print("     [WARN] Config missing -- marking all synthesizable")
        return {s: True for s in candidates}, len(candidates)
    synth: dict = {}
    _pending = list(candidates)
    _to = SYNTH_TIMEOUT or None
    for _round in range(1, 99):
        if _round > 1:
            print(f"     [restart] round {_round}, {len(_pending)} remaining")
        with ProcessPoolExecutor(
            max_workers=N_WORKERS,
            initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),),
            max_tasks_per_child=50,
        ) as pool:
            futures = {pool.submit(score_one, s): s for s in _pending}
            broken  = False
            with tqdm(total=len(_pending),
                      desc=f"AiZynthFinder (round {_round})", unit="mol") as pbar:
                for fut in as_completed(futures):
                    smi = futures[fut]
                    try:
                        _, solved = fut.result(timeout=_to)
                        pbar.update(1)
                    except FutureTimeoutError:
                        solved = False
                    except BrokenProcessPool:
                        broken = True; break
                    except Exception:
                        solved = False
                    synth[smi] = solved
            if broken:
                pass
        _pending = [s for s in candidates if s not in synth]
        if not _pending:
            break
    with open(ckpt_path, "w") as f:
        json.dump(synth, f)
    n_solved = sum(synth.values())
    print(f"     Solved: {n_solved}/{len(synth)} "
          f"({100 * n_solved / max(len(synth), 1):.1f}%)")
    return synth, len(candidates)

print("Imports OK | synth_gate_4a + synth_gate_4b defined")

Imports OK | synth_gate_4a + synth_gate_4b defined


In [3]:
# GPU availability check for JT-VAE backend
import subprocess, sys

_jtvae_py = os.environ.get("JT_VAE_PYTHON", "")
if _jtvae_py and Path(_jtvae_py).exists():
    _result = subprocess.run(
        [_jtvae_py, "-c",
         "import torch; "
         "print('torch:', torch.__version__); "
         "print('CUDA available:', torch.cuda.is_available()); "
         "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')"
        ],
        capture_output=True, text=True
    )
    print("[JT-VAE env]")
    print(_result.stdout.strip())
    if _result.stderr:
        print("WARN:", _result.stderr.strip()[:200])
else:
    print("JT_VAE_PYTHON not set or not found — JT-VAE will be skipped")

[JT-VAE env]
torch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti


---
## Stage 1 — ChEMBL Features

Load pre-computed fingerprints and descriptors from `chembl_features.npz`.
Generate the file if missing with:
```
python -m src.generation.main featurize
```

In [4]:
assert CHEMBL_NPZ.exists(), (
    f"chembl_features.npz not found: {CHEMBL_NPZ}\n"
    "Run: python -m src.generation.main featurize")

_data = np.load(CHEMBL_NPZ, allow_pickle=True)

# Subset to N_SPECS molecules
_idx = np.arange(min(N_SPECS, len(_data["smiles"])))
smiles_arr     = _data["smiles"][_idx]             # (N,)  object
ecfp4_arr      = _data["ecfp4"][_idx]              # (N, 2048)
fcfp4_arr      = _data["fcfp4"][_idx]              # (N, 2048)
rdkit_vals     = _data["rdkit_desc_values"][_idx]  # (N, D)
rdkit_names    = _data["rdkit_desc_names"].tolist()
brics_fps      = _data["brics_fps"][_idx]          # (N, 8, 2048)
brics_exists   = _data["brics_exists"][_idx]       # (N, 8)

# Lookup by SMILES so Stage 2 & 3 are order-independent
feat = {
    smi: {
        "ecfp4":  ecfp4_arr[i],
        "fcfp4":  fcfp4_arr[i],
        "rdkit":  rdkit_vals[i],
        "brics":  brics_fps[i],
        "bric_e": brics_exists[i],
    }
    for i, smi in enumerate(smiles_arr)
}

print(f"Loaded {len(smiles_arr)} molecules from {CHEMBL_NPZ.name}")
print(f"ecfp4={ecfp4_arr.shape}  brics_fps={brics_fps.shape}")

Loaded 100 molecules from chembl_features.npz
ecfp4=(100, 2048)  brics_fps=(100, 8, 2048)


---
## Stage 2 — PrexSyn Seeds

For each spec, POST feature vectors to the PrexSyn API, retrieve
`N_PREXSYN_SAMPLES` candidates, and keep the one with highest ECFP4
Tanimoto to the spec.  Saves `seeds_for_methods.json`.

In [5]:
def _quality_bin(bq: float) -> str:
    if   bq < 0.50: return "<0.5"
    elif bq < 0.70: return "0.5-0.7"
    elif bq < 0.85: return "0.7-0.85"
    else:           return "0.85-1.0"


if USE_CACHED_SEEDS and SEEDS_JSON.exists() and SEEDS_JSON.stat().st_size > 0:
    with open(SEEDS_JSON) as f:
        seeds = json.load(f)
    print(f"Loaded {len(seeds)} seeds from cache ({SEEDS_JSON.name})")
else:
    seeds = []
    api_errors = []
    for smi in tqdm(smiles_arr, desc="PrexSyn seeds"):
        smi = str(smi)
        f   = feat[smi]
        payload = {
            "ecfp4":             f["ecfp4"].tolist(),
            "fcfp4":             f["fcfp4"].tolist(),
            "rdkit_desc_values": f["rdkit"].tolist(),
            "rdkit_desc_names":  rdkit_names,
            "brics_fps":         f["brics"].tolist(),
            "brics_exists":      f["bric_e"].tolist(),
            "source_smiles":     smi,
            "num_samples":       N_PREXSYN_SAMPLES,
        }
        try:
            resp       = requests.post(PREXSYN_URL, json=payload, timeout=300)
            resp.raise_for_status()
            candidates = resp.json().get("generated_smiles", [])
        except Exception as e:
            api_errors.append(f"{smi[:40]}: {e}")
            candidates = []

        spec = make_spec(smi)
        best_smi, best_t = smi, 0.0
        if spec and candidates:
            for c in candidates:
                if "." in c:
                    continue
                mol = Chem.MolFromSmiles(c)
                if mol:
                    t = tanimoto_to_spec(mol, spec)
                    if t > best_t:
                        best_t, best_smi = t, c

        seeds.append({
            "spec_smiles":      smi,
            "seed_smiles":      best_smi,
            "baseline_quality": round(best_t, 4),
            "quality_bin":      _quality_bin(best_t),
            "methods":          {},
        })

    with open(SEEDS_JSON, "w") as f:
        json.dump(seeds, f, indent=2)
    print(f"Seeds saved -> {SEEDS_JSON}")
    if api_errors:
        print(f"\n⚠️  {len(api_errors)} API errors occurred:")
        for err in api_errors[:5]:
            print(f"   {err}")
        if len(api_errors) > 5:
            print(f"   ... and {len(api_errors) - 5} more")

# Build lookup maps used by all downstream stages
spec_to_seed = {e["spec_smiles"]: e["seed_smiles"]      for e in seeds}
spec_to_bq   = {e["spec_smiles"]: e["baseline_quality"] for e in seeds}
specs_list   = [e["spec_smiles"] for e in seeds]
seeds_list   = [e["seed_smiles"] for e in seeds]

from collections import Counter
bin_counts = Counter(e["quality_bin"] for e in seeds)
print(f"\nQuality distribution: {dict(bin_counts)}")
print(f"Median baseline_quality: {np.median([e['baseline_quality'] for e in seeds]):.3f}")

Loaded 100 seeds from cache (seeds_for_methods.json)

Quality distribution: {'0.5-0.7': 31, '0.85-1.0': 25, '<0.5': 33, '0.7-0.85': 11}
Median baseline_quality: 0.581


---
## Stage 3 — PrexSyn Resampling (Baseline)

Call PrexSyn again for `N_BASELINE_RESAMP` samples per spec.  Score each
variant with `scoring_v2.score_batch()`.  Then apply the Stage 4b gate.

In [6]:
if USE_CACHED_BASELINE and BASELINE_SCORES.exists() and BASELINE_SCORES.stat().st_size > 0:
    df_baseline = pd.read_csv(BASELINE_SCORES)
    print(f"Loaded {len(df_baseline):,} rows from cache ({BASELINE_SCORES.name})")
else:
    _rows = []
    for smi in tqdm(specs_list, desc="Baseline resample"):
        f    = feat.get(smi)
        spec = make_spec(smi)
        if f is None or spec is None:
            continue
        payload = {
            "ecfp4":             f["ecfp4"].tolist(),
            "fcfp4":             f["fcfp4"].tolist(),
            "rdkit_desc_values": f["rdkit"].tolist(),
            "rdkit_desc_names":  rdkit_names,
            "brics_fps":         f["brics"].tolist(),
            "brics_exists":      f["bric_e"].tolist(),
            "source_smiles":     smi,
            "num_samples":       N_BASELINE_RESAMP,
        }
        try:
            resp     = requests.post(PREXSYN_URL, json=payload, timeout=300)
            resp.raise_for_status()
            variants = resp.json().get("generated_smiles", [])
        except Exception as e:
            print(f"  [WARN] {smi[:40]}: {e}")
            variants = []

        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[smi],
            method="baseline",
        ))

    df_baseline = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_baseline.to_csv(BASELINE_SCORES, index=False)
    print(f"Saved -> {BASELINE_SCORES}  ({len(df_baseline):,} rows)")

Loaded 6,005 rows from cache (baseline_scores.csv)


In [7]:
# Stage 4b gate for baseline: score first, AiZynthFinder on property-passing only
synth_baseline, n_eval_baseline = synth_gate_4b(df_baseline, BASELINE_CKPT)
df_baseline["is_synth"] = (
    df_baseline["variant_smiles"].map(synth_baseline).fillna(False))
_hits = classify_hits(df_baseline, TAU_T_LIST[0], TAU_D) & df_baseline["is_synth"]
print(f"Baseline hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {n_eval_baseline} evaluated (total rows: {len(df_baseline)})")

[4b] SKIP_SYNTH_CHECK=True: marking 78 passing property gate as synthesizable
Baseline hits (tau_t=0.6, tau_d=0.5): 78 / 78 evaluated (total rows: 6005)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_17092\3164473216.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_baseline["variant_smiles"].map(synth_baseline).fillna(False))


---
## Stage 4 — CReM Modification

Apply context-dependent fragment substitution (CReM) to each seed.
Requires `replacements02_sc2.db` (set `CREM_DB_PATH` or update `CREM_DB` above).
Download from: <https://zenodo.org/record/4519690>

In [8]:
from src.modifications.rule_based.crem_modifier import CRemModifier

if USE_CACHED_CREM and CREM_SCORES.exists() and CREM_SCORES.stat().st_size > 0:
    df_crem = pd.read_csv(CREM_SCORES)
    print(f"Loaded {len(df_crem):,} rows from cache ({CREM_SCORES.name})")
else:
    if not CREM_DB.exists():
        raise FileNotFoundError(
            f"CReM database not found: {CREM_DB}\n"
            "Download chembl33_sa2_f5.db from https://zenodo.org/records/16909329")

    crem = CRemModifier(db_path=CREM_DB)
    _rows = []
    for spec_smi, seed_smi in tqdm(
            list(zip(specs_list, seeds_list)), desc="CReM"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        try:
            variants = crem.modify(seed_smi, N_CREM_VARIANTS)
        except Exception as e:
            print(f"  [WARN] {seed_smi[:40]}: {e}")
            variants = []
        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[spec_smi],
            method="CReM",
        ))

    df_crem = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_crem.to_csv(CREM_SCORES, index=False)
    print(f"Saved -> {CREM_SCORES}  ({len(df_crem):,} rows)")

Loaded 10,689 rows from cache (crem_scores.csv)


In [9]:
# Stage 4b gate for CReM
synth_crem, n_eval_crem = synth_gate_4b(df_crem, CREM_CKPT)
df_crem["is_synth"] = df_crem["variant_smiles"].map(synth_crem).fillna(False)
_hits = classify_hits(df_crem, TAU_T_LIST[0], TAU_D) & df_crem["is_synth"]
print(f"CReM hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {n_eval_crem} evaluated (total rows: {len(df_crem)})")

[4b] SKIP_SYNTH_CHECK=True: marking 294 passing property gate as synthesizable
CReM hits (tau_t=0.6, tau_d=0.5): 294 / 294 evaluated (total rows: 10689)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_17092\1841178918.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_crem["is_synth"] = df_crem["variant_smiles"].map(synth_crem).fillna(False)


---
## Stage 5 — LibINVENT Decoration

Decompose each seed into BRICS scaffolds, then decorate with the
pretrained LibINVENT RNN (runs as a subprocess in `libinvent_env`).

In [10]:
from fragment import get_scaffolds

# Extract BRICS scaffolds from each seed
scaffold_map: dict[str, list[str]] = {}   # spec_smi -> [scaffold_smi, ...]
_all_sc: list[str] = []
_seen   = set()

for spec_smi, seed_smi in tqdm(
        list(zip(specs_list, seeds_list)), desc="BRICS scaffolds"):
    scs = get_scaffolds(seed_smi, method="brics")
    scaffold_map[spec_smi] = scs
    for sc in scs:
        if sc not in _seen:
            _seen.add(sc)
            _all_sc.append(sc)

print(f"Seeds: {len(specs_list)} | Unique scaffolds: {len(_all_sc)}")

BRICS scaffolds:   0%|          | 0/100 [00:00<?, ?it/s]

[fragment] Warning: no fragments found for 'Cc1nn(C)c(=O)[nH]c1=O'. Returning original molecule as a placeholder scaffold.
[fragment] Warning: no fragments found for 'NC1=NS(=O)(=O)Nc2ccccc21'. Returning original molecule as a placeholder scaffold.
Seeds: 100 | Unique scaffolds: 197


In [11]:
from configs import make_decorate_config

# Scaffold canonicalization -- BRICS labels [*:0],[*:1]; LibINVENT outputs [*]
_ATTACH = re.compile(r'\[\*(?::\d+)?\]')

def _canon_sc(smi: str):
    mol = Chem.MolFromSmiles(re.sub(r'\[\*(?::\d+)?\]', '[*]', smi))
    if mol is None:
        return None
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol)


# Build canonical scaffold -> [spec_smi, ...] reverse map
_sc_to_specs: dict[str, list[str]] = {}
for spec_smi, scs in scaffold_map.items():
    for sc in scs:
        key = _canon_sc(sc)
        if key:
            _sc_to_specs.setdefault(key, []).append(spec_smi)

# ── Run LibINVENT if output not cached ────────────────────────────────────────
if USE_CACHED_LI and LI_SCORES.exists() and LI_SCORES.stat().st_size > 0:
    df_li = pd.read_csv(LI_SCORES)
    print(f"Loaded {len(df_li):,} rows from cache ({LI_SCORES.name})")
else:
    assert LIB_INVENT_MODEL.exists(), f"Model not found: {LIB_INVENT_MODEL}"
    assert LIB_INVENT_PY.exists(),    f"input.py not found: {LIB_INVENT_PY}"

    # Write scaffolds.smi
    LI_SCAFFOLDS_SMI.write_text("\n".join(_all_sc))
    print(f"Wrote {len(_all_sc)} scaffolds -> {LI_SCAFFOLDS_SMI}")

    # Write decorate_config.json
    cfg = make_decorate_config(
        model_path         = str(LIB_INVENT_MODEL.resolve()),
        scaffolds_smi_path = str(LI_SCAFFOLDS_SMI.resolve()),
        output_csv_path    = str(LI_DECORATED_CSV.resolve()),
        logging_path       = str(LI_LOG_DIR.resolve()),
        batch_size         = 64,
        n_decorations      = N_LI_DECORATIONS,
    )
    LI_CONFIG_JSON.write_text(json.dumps(cfg, indent=2))

    # Run LibINVENT with the current kernel's Python (same env as this notebook)
    import sys as _sys
    cmd = [_sys.executable,
           str(LIB_INVENT_PY.resolve()),
           str(LI_CONFIG_JSON.resolve())]
    print(f"Running: {' '.join(cmd)}\n")
    result = subprocess.run(
        cmd, cwd=str(LIB_INVENT_ROOT),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout or "")
    if result.returncode != 0:
        raise RuntimeError(
            f"LibINVENT exited with code {result.returncode}")

    # Parse decorated.csv and score
    assert LI_DECORATED_CSV.exists(), "decorated.csv not produced by LibINVENT"
    df_dec = pd.read_csv(LI_DECORATED_CSV)
    df_dec = df_dec[df_dec["SMILES"].apply(
        lambda s: Chem.MolFromSmiles(str(s)) is not None
    )].copy()
    df_dec["_canon_sc"] = df_dec["Scaffold"].apply(_canon_sc)

    # Score each variant against its matching spec(s)
    _nll_map = dict(zip(df_dec["SMILES"], df_dec.get("Likelihoods", [None]*len(df_dec))))
    _rows = []
    for spec_smi in tqdm(specs_list, desc="Score LibINVENT"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        # Variants whose scaffold came from this spec's seed
        _my_scs = {_canon_sc(sc) for sc in scaffold_map.get(spec_smi, [])}
        _mask   = df_dec["_canon_sc"].isin(_my_scs)
        variants = df_dec.loc[_mask, "SMILES"].tolist()
        if not variants:
            continue
        df_s = score_batch(variants, spec,
                           baseline_quality=spec_to_bq[spec_smi],
                           method="LibINVENT")
        df_s["nll"] = df_s["variant_smiles"].map(_nll_map)
        _rows.append(df_s)

    df_li = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_li.to_csv(LI_SCORES, index=False)
    print(f"Saved -> {LI_SCORES}  ({len(df_li):,} rows)")

Loaded 32,273 rows from cache (libinvent_scores.csv)


In [12]:
# Stage 4b gate for LibINVENT
synth_li, n_eval_li = synth_gate_4b(df_li, LI_CKPT)
df_li["is_synth"] = df_li["variant_smiles"].map(synth_li).fillna(False)
_hits = classify_hits(df_li, TAU_T_LIST[0], TAU_D) & df_li["is_synth"]
print(f"LibINVENT hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {n_eval_li} evaluated (total rows: {len(df_li)})")

[4b] SKIP_SYNTH_CHECK=True: marking 14 passing property gate as synthesizable
LibINVENT hits (tau_t=0.6, tau_d=0.5): 14 / 14 evaluated (total rows: 32273)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_17092\466560035.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_li["is_synth"] = df_li["variant_smiles"].map(synth_li).fillna(False)


---
## Stage 6 — mmpdb Modification

Apply matched molecular pair (MMP) transforms to each seed using a prebuilt mmpdb database.
Each seed is transformed with `mmpdb transform`, applying A→B substitution rules learned
from ChEMBL. Requires `data/mmpdb_db/chembl_50k.mmpdb` (build with `python scripts/setup_mmpdb_db.py`).

In [13]:
from src.modifications.rule_based.mmpdb_modifier import MmpdbModifier

if USE_CACHED_MMPDB and MMPDB_SCORES.exists() and MMPDB_SCORES.stat().st_size > 0:
    df_mmpdb = pd.read_csv(MMPDB_SCORES)
    print(f"Loaded {len(df_mmpdb):,} rows from cache ({MMPDB_SCORES.name})")
else:
    mmpdb = MmpdbModifier(db_path=MMPDB_DB)
    _rows = []
    for spec_smi, seed_smi in tqdm(
            list(zip(specs_list, seeds_list)), desc="mmpdb"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        try:
            variants = mmpdb.modify(seed_smi, N_MMPDB_VARIANTS)
        except Exception as e:
            print(f"  [WARN] {seed_smi[:40]}: {e}")
            variants = []
        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[spec_smi],
            method="mmpdb",
        ))

    df_mmpdb = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_mmpdb.to_csv(MMPDB_SCORES, index=False)
    print(f"Saved -> {MMPDB_SCORES}  ({len(df_mmpdb):,} rows)")

Loaded 1,849 rows from cache (mmpdb_scores.csv)


In [14]:
# Stage 4b gate for mmpdb
synth_mmpdb, n_eval_mmpdb = synth_gate_4b(df_mmpdb, MMPDB_CKPT)
df_mmpdb["is_synth"] = df_mmpdb["variant_smiles"].map(synth_mmpdb).fillna(False)
_hits = classify_hits(df_mmpdb, TAU_T_LIST[0], TAU_D) & df_mmpdb["is_synth"]
print(f"mmpdb hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {n_eval_mmpdb} evaluated (total rows: {len(df_mmpdb)})")

[4b] SKIP_SYNTH_CHECK=True: marking 101 passing property gate as synthesizable
mmpdb hits (tau_t=0.6, tau_d=0.5): 101 / 101 evaluated (total rows: 1849)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_17092\3592569490.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_mmpdb["is_synth"] = df_mmpdb["variant_smiles"].map(synth_mmpdb).fillna(False)


---
## Stage 7 — JT-VAE Modification

Encode each seed into the JT-VAE latent space, add Gaussian noise, and decode to produce
novel variants. The backend runs in an isolated subprocess (`backend_infer.py`) using a
separate virtual environment (`JT_VAE_PYTHON`).

**Setup:** run `bash scripts/setup_jt_vae_env.sh` and set the environment variables:
- `JT_VAE_PYTHON` — path to the venv Python interpreter
- `JT_VAE_MODEL_PATH` — path to the pretrained `.pkl` checkpoint

In [ ]:
from src.modifications.ml_based.jt_vae import JTVAEModifier

JTVAE_NUM_WORKERS = 6  # CPU parallel threads; tune up/down based on CPU usage
JTVAE_CACHE       = JTVAE_DIR / "jtvae_cache.json"

if USE_CACHED_JTVAE and JTVAE_SCORES.exists() and JTVAE_SCORES.stat().st_size > 0:
    df_jtvae = pd.read_csv(JTVAE_SCORES)
    print(f"Loaded {len(df_jtvae):,} rows from cache ({JTVAE_SCORES.name})")
else:
    jtvae = JTVAEModifier(noise_scale=0.30, attempts_per_variant=4)

    # Batch call: load model once for all seeds, decode in parallel threads.
    # Results are saved to JTVAE_CACHE after each seed so a crash can be resumed.
    print(f"Running JT-VAE batch on {len(seeds_list)} seeds "
          f"(device=cpu, num_workers={JTVAE_NUM_WORKERS})...")
    batch_results = jtvae.modify_batch(
        seeds_list, N_JTVAE_VARIANTS,
        num_workers=JTVAE_NUM_WORKERS,
        cache_path=JTVAE_CACHE,
    )

    _rows = []
    for spec_smi, seed_smi in tqdm(list(zip(specs_list, seeds_list)), desc="JT-VAE scoring"):
        spec = make_spec(spec_smi)
        if spec is None:
            continue
        variants = batch_results.get(seed_smi, [])
        _rows.append(score_batch(
            variants, spec,
            baseline_quality=spec_to_bq[spec_smi],
            method="JT-VAE",
        ))

    df_jtvae = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame()
    df_jtvae.to_csv(JTVAE_SCORES, index=False)
    print(f"Saved -> {JTVAE_SCORES}  ({len(df_jtvae):,} rows)")


Running JT-VAE batch on 100 seeds (device=cpu, num_workers=6)...
JT-VAE cache: 26/100 seeds already cached, running 74 new seeds.
C:\Users\Acmaro\anaconda3\envs\jtvae_env\lib\site-packages\torch\nn\modules\loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)
[JT-VAE] SKIP 'CCN(CC)S(=O)(=O)N(c1nccn1Cc1cccnc1)C1CN2CCCC1C2': fragment not in ZINC vocab ('C1CC2CCN(C1)C2')
[JT-VAE] 1/74 CCN(CC)S(=O)(=O)N(c1nccn1Cc1cccnc1)C1CN2


In [ ]:
# Stage 4b gate for JT-VAE
synth_jtvae, n_eval_jtvae = synth_gate_4b(df_jtvae, JTVAE_CKPT)
df_jtvae["is_synth"] = df_jtvae["variant_smiles"].map(synth_jtvae).fillna(False)
_hits = classify_hits(df_jtvae, TAU_T_LIST[0], TAU_D) & df_jtvae["is_synth"]
print(f"JT-VAE hits (tau_t={TAU_T_LIST[0]}, tau_d={TAU_D}): "
      f"{_hits.sum()} / {n_eval_jtvae} evaluated (total rows: {len(df_jtvae)})")

---
## Outputs

All files consumed by `pipeline_analysis.ipynb`.

In [ ]:
outputs = [
    ("seeds_for_methods.json", SEEDS_JSON),
    ("baseline_scores.csv",    BASELINE_SCORES),
    ("baseline ckpt",          BASELINE_CKPT),
    ("crem_scores.csv",        CREM_SCORES),
    ("crem ckpt",              CREM_CKPT),
    ("libinvent_scores.csv",   LI_SCORES),
    ("libinvent ckpt",         LI_CKPT),
    ("mmpdb_scores.csv",       MMPDB_SCORES),
    ("mmpdb ckpt",             MMPDB_CKPT),
    ("jtvae_scores.csv",       JTVAE_SCORES),
    ("jtvae ckpt",             JTVAE_CKPT),
]
print(f"{'File':<28}  {'Size':>8}  {'Status'}")
print("-" * 55)
for label, path in outputs:
    ok   = path.exists()
    size = f"{path.stat().st_size/1e3:.1f} KB" if ok else "-"
    print(f"{label:<28}  {size:>8}  {'OK' if ok else 'MISSING'}")

print()
print("Add to pipeline_analysis.ipynb METHODS dict:")
print(f'  "baseline" : {{')
print(f'      "scores":     DATA_DIR / "generation/baseline/baseline_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/baseline/synth_checkpoint.json",')
print(f'  }},')
print(f'  "CReM" : {{')
print(f'      "scores":     DATA_DIR / "generation/crem/crem_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/crem/synth_checkpoint.json",')
print(f'  }},')
print(f'  "LibINVENT" : {{')
print(f'      "scores":     DATA_DIR / "generation/libinvent/libinvent_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/libinvent/synth_checkpoint.json",')
print(f'  }},')
print(f'  "mmpdb" : {{')
print(f'      "scores":     DATA_DIR / "generation/mmpdb/mmpdb_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/mmpdb/synth_checkpoint.json",')
print(f'  }},')
print(f'  "JT-VAE" : {{')
print(f'      "scores":     DATA_DIR / "generation/jtvae/jtvae_scores.csv",')
print(f'      "synth_ckpt": DATA_DIR / "generation/jtvae/synth_checkpoint.json",')
print(f'  }},')